# Behavior-Aware SASRec

Train and compare:
- **purchase_only**: SASRec on purchase sequences only
- **behavior_aware**: same SASRec backbone + cart add/remove behavior embeddings

This notebook aligns with the shared baseline evaluation protocol from
`Step1_SequenceConstruction.ipynb` and `Step2_CandidateSampling_PopularityBaseline.ipynb`:

- same `MAX_SEQ_LEN` / 14-day val / 14-day test cutoffs
- same `item_to_idx` vocabulary
- same saved candidate sets (100 negatives + 1 target, original product IDs)
- shared metrics via `evaluation_utils.py` (`HitRate@10`, `NDCG@10`, `MRR`)

Prediction scores are produced in the **same order** as each user's candidate list.


## 0. Setup


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence
import json
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from evaluation_utils import evaluate_model_dataset

ARTIFACT_DIR = Path("artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| device:", DEVICE)
print("Artifact dir:", ARTIFACT_DIR.resolve())

## 1. Config

Set `USE_SHARED_ARTIFACTS = True` for the real shared eval (loads Step1/Step2 pickles).
Set `USE_SHARED_ARTIFACTS = False` for a tiny synthetic dry-run of the same protocol.


In [ ]:
# --- protocol / data ---
USE_SHARED_ARTIFACTS = True   # False => synthetic mini shared-style split
MODE = "both"                 # "purchase_only" | "behavior_aware" | "both"
EVAL_SPLIT = "val"            # "val" during development; "test" for final numbers only

# Cart parquet locations (needed for behavior_aware when USE_SHARED_ARTIFACTS=True)
DATA_CANDIDATES = [
    Path("data/subsampled"),
    Path("/content/subsampled"),
    Path("Desktop/DS5500/data/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]

# --- model / train ---
HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 3 if not USE_SHARED_ARTIFACTS else 10
N_NEGATIVES_TRAIN = 1
PATIENCE = 3
SEED = 42
# Optional speed knob for local iteration on the full cohort (None = all train users)
MAX_TRAIN_USERS = None if USE_SHARED_ARTIFACTS else None

OUTPUT_DIR = ARTIFACT_DIR / ("sasrec_shared_synthetic" if not USE_SHARED_ARTIFACTS else "sasrec_shared_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("USE_SHARED_ARTIFACTS =", USE_SHARED_ARTIFACTS)
print("OUTPUT_DIR =", OUTPUT_DIR)

## 2. Load shared sequences, vocabulary, and candidate sets


In [ ]:
def build_synthetic_shared_artifacts(n_users=80, n_items=250, seed=42):
    """Mini artifacts with the same schema as Step1/Step2 for dry-runs."""
    rng = np.random.default_rng(seed)
    all_items = list(range(1, n_items + 1))
    item_to_idx = {sku: i + 1 for i, sku in enumerate(all_items)}
    idx_to_item = {i: sku for sku, i in item_to_idx.items()}

    train_sequences, val_targets, test_targets = {}, {}, {}
    for u in range(1, n_users + 1):
        L = int(rng.integers(3, 12))
        seq = rng.choice(all_items, size=L, replace=True).tolist()
        train_sequences[u] = seq
        val_targets[u] = int(rng.choice(all_items))
        test_targets[u] = int(rng.choice(all_items))

    val_eval_users = list(train_sequences.keys())
    test_eval_users = list(train_sequences.keys())

    def sample_cands(users, targets, seed_offset):
        local_rng = np.random.default_rng(seed + seed_offset)
        out = {}
        pool = np.asarray(all_items)
        for u in users:
            target = targets[u]
            negs = local_rng.choice(pool, size=100, replace=False).tolist()
            if target in negs:
                negs = [x for x in negs if x != target]
                while len(negs) < 100:
                    x = int(local_rng.choice(pool))
                    if x != target and x not in negs:
                        negs.append(x)
            cands = negs[:100] + [target]
            local_rng.shuffle(cands)
            out[u] = cands
        return out

    sequences = {
        "config": {
            "MAX_SEQ_LEN": 50,
            "TEST_DAYS": 14,
            "VAL_DAYS": 14,
            "val_cutoff": pd.Timestamp("2022-11-10"),
            "test_cutoff": pd.Timestamp("2022-11-24"),
        },
        "train_sequences": train_sequences,
        "val_targets": val_targets,
        "test_targets": test_targets,
        "val_eval_users": val_eval_users,
        "test_eval_users": test_eval_users,
        "item_to_idx": item_to_idx,
        "idx_to_item": idx_to_item,
        "n_items": len(all_items),
        "all_items": all_items,
    }
    candidates = {
        "val_candidate_sets": sample_cands(val_eval_users, val_targets, 1),
        "test_candidate_sets": sample_cands(test_eval_users, test_targets, 2),
        "n_negatives": 100,
        "eval_k": 10,
        "random_seed": seed,
        "candidate_pool": "full_item_vocabulary",
        "include_unseen_targets": True,
        "max_seq_len": 50,
        "val_cutoff": sequences["config"]["val_cutoff"],
        "test_cutoff": sequences["config"]["test_cutoff"],
    }
    return sequences, candidates


if USE_SHARED_ARTIFACTS:
    seq_path = ARTIFACT_DIR / "sequences_and_splits.pkl"
    cand_path = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"
    if not seq_path.exists() or not cand_path.exists():
        raise FileNotFoundError(
            "Missing shared artifacts. Expected:\n"
            f"  {seq_path}\n  {cand_path}\n"
            "Run Step1_SequenceConstruction.ipynb and "
            "Step2_CandidateSampling_PopularityBaseline.ipynb first."
        )
    with seq_path.open("rb") as f:
        shared = pickle.load(f)
    with cand_path.open("rb") as f:
        shared_cands = pickle.load(f)
    print("Loaded shared Step1/Step2 artifacts")
else:
    shared, shared_cands = build_synthetic_shared_artifacts(seed=SEED)
    print("Built synthetic shared-style artifacts")

train_sequences = shared["train_sequences"]
val_targets = shared["val_targets"]
test_targets = shared["test_targets"]
val_eval_users = shared["val_eval_users"]
test_eval_users = shared["test_eval_users"]
item_to_idx = shared["item_to_idx"]
idx_to_item = shared["idx_to_item"]
all_items = shared["all_items"]
n_items = int(shared["n_items"])
config = shared["config"]

MAX_SEQ_LEN = int(config["MAX_SEQ_LEN"])
VAL_CUTOFF = pd.Timestamp(config["val_cutoff"])
TEST_CUTOFF = pd.Timestamp(config["test_cutoff"])
EVAL_K = int(shared_cands.get("eval_k", 10))
N_NEGATIVES_EVAL = int(shared_cands.get("n_negatives", 100))

val_candidate_sets = shared_cands["val_candidate_sets"]
test_candidate_sets = shared_cands["test_candidate_sets"]

print(f"MAX_SEQ_LEN={MAX_SEQ_LEN} | VAL_CUTOFF={VAL_CUTOFF} | TEST_CUTOFF={TEST_CUTOFF}")
print(f"n_items={n_items:,} | train users={len(train_sequences):,}")
print(f"val_eval_users={len(val_eval_users):,} | test_eval_users={len(test_eval_users):,}")
print(f"candidates/user={N_NEGATIVES_EVAL + 1} | EVAL_K={EVAL_K}")

# Sanity: shared candidate protocol
assert set(val_candidate_sets) == set(val_eval_users)
assert set(test_candidate_sets) == set(test_eval_users)
_u = val_eval_users[0]
assert len(val_candidate_sets[_u]) == N_NEGATIVES_EVAL + 1
assert val_targets[_u] in val_candidate_sets[_u]
print("Shared candidate checks passed")


## 3. Build training histories

- **purchase_only** uses `train_sequences` from the shared pickle (purchases before `val_cutoff`).
- **behavior_aware** loads buy + cart events with `timestamp < val_cutoff` and maps SKUs through the **shared** `item_to_idx` (cart-only SKUs missing from the purchase vocab are dropped).


In [ ]:
BEHAVIOR_PAD = 0
BEHAVIOR_BUY = 1
BEHAVIOR_ADD = 2
BEHAVIOR_REMOVE = 3

BEHAVIOR_NAME_TO_ID = {
    "product_buy": BEHAVIOR_BUY,
    "add_to_cart": BEHAVIOR_ADD,
    "remove_from_cart": BEHAVIOR_REMOVE,
}


@dataclass(frozen=True)
class BehaviorEvent:
    item_id: int          # encoded index (shared item_to_idx)
    behavior_id: int
    raw_item: int         # original product ID


def purchase_histories_from_shared(train_sequences, item_to_idx, max_len):
    """user -> list[BehaviorEvent] using shared purchase sequences."""
    out = {}
    for user, seq in train_sequences.items():
        events = []
        for raw in seq[-max_len:]:
            if raw not in item_to_idx:
                continue
            events.append(BehaviorEvent(item_to_idx[raw], BEHAVIOR_BUY, int(raw)))
        if events:
            out[int(user)] = events
    return out


def find_data_dir():
    return next((p for p in DATA_CANDIDATES if p.exists()), None)


def load_multibehavior_train_histories(data_dir, item_to_idx, val_cutoff, max_len):
    """Build cart-aware histories strictly before val_cutoff with shared IDs."""
    frames = []
    file_map = {
        "product_buy": "product_buy_subsampled.parquet",
        "add_to_cart": "add_to_cart_subsampled.parquet",
        "remove_from_cart": "remove_from_cart_subsampled.parquet",
    }
    for behavior, fname in file_map.items():
        path = Path(data_dir) / fname
        if not path.exists():
            raise FileNotFoundError(path)
        df = pd.read_parquet(path).drop_duplicates().copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp", "client_id", "sku"])
        df["behavior"] = behavior
        frames.append(df[["client_id", "sku", "timestamp", "behavior"]])

    events = pd.concat(frames, ignore_index=True)
    events = events[events["timestamp"] < val_cutoff].copy()
    events = events[events["sku"].isin(item_to_idx.keys())].copy()
    events["item_id"] = events["sku"].map(item_to_idx).astype(np.int64)
    events["behavior_id"] = events["behavior"].map(BEHAVIOR_NAME_TO_ID).astype(np.int64)
    events = events.sort_values(["client_id", "timestamp"]).reset_index(drop=True)

    histories = {}
    for user, group in events.groupby("client_id", sort=False):
        rows = list(group.itertuples(index=False))
        rows = rows[-max_len:]
        histories[int(user)] = [
            BehaviorEvent(int(r.item_id), int(r.behavior_id), int(r.sku))
            for r in rows
        ]
    return histories


def make_synthetic_cart_histories(purchase_histories, seed=42):
    """Augment purchase histories with synthetic cart events for dry-runs."""
    rng = np.random.default_rng(seed)
    out = {}
    for user, buys in purchase_histories.items():
        events = []
        for ev in buys:
            if rng.random() < 0.7:
                events.append(BehaviorEvent(ev.item_id, BEHAVIOR_ADD, ev.raw_item))
            events.append(ev)
        out[user] = events[-MAX_SEQ_LEN:]
    return out


purchase_histories = purchase_histories_from_shared(train_sequences, item_to_idx, MAX_SEQ_LEN)
print(f"purchase_only histories: {len(purchase_histories):,}")

behavior_histories = None
if MODE in ("behavior_aware", "both"):
    if not USE_SHARED_ARTIFACTS:
        behavior_histories = make_synthetic_cart_histories(purchase_histories, seed=SEED)
        print(f"behavior_aware histories (synthetic cart): {len(behavior_histories):,}")
    else:
        data_dir = find_data_dir()
        if data_dir is None:
            print(
                "WARNING: no subsampled parquet found; behavior_aware training/eval "
                "histories unavailable. purchase_only can still run from the shared pickle."
            )
        else:
            print("Loading cart+buy events from:", data_dir.resolve())
            behavior_histories = load_multibehavior_train_histories(
                data_dir, item_to_idx, VAL_CUTOFF, MAX_SEQ_LEN
            )
            # Keep users that also have a shared purchase train sequence
            behavior_histories = {
                u: h for u, h in behavior_histories.items() if u in purchase_histories and h
            }
            print(f"behavior_aware histories: {len(behavior_histories):,}")

## 4. Model + training helpers


In [ ]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_size: int, dropout: float) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout1 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x.transpose(1, 2)
        out = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(out)))))
        return out.transpose(1, 2)


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = PointWiseFeedForward(hidden_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False
        )
        x = x + self.dropout(attn_out)
        h = self.norm2(x)
        return x + self.ffn(h)


class SASRec(nn.Module):
    def __init__(self, n_items, hidden_size=64, max_len=50, n_blocks=2, n_heads=1, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.item_emb = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([SASRecBlock(hidden_size, n_heads, dropout) for _ in range(n_blocks)])
        self.final_norm = nn.LayerNorm(hidden_size, eps=1e-8)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding, nn.Conv1d)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].fill_(0)
            if isinstance(module, nn.LayerNorm):
                nn.init.zeros_(module.bias)
                nn.init.ones_(module.weight)

    def _causal_mask(self, seq_len, device):
        return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(self, item_ids, behavior_ids=None):
        del behavior_ids
        bsz, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(bsz, -1)
        x = self.emb_dropout(self.item_emb(item_ids) + self.pos_emb(positions))
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)

    def predict(self, item_ids, behavior_ids=None, candidate_items=None):
        # Histories are LEFT-padded, so the most recent event is always at the
        # final timestep (index -1). Using (nonzero count - 1) would incorrectly
        # point at the first real item.
        seq_emb = self.encode(item_ids, behavior_ids)
        final = seq_emb[:, -1, :]
        if candidate_items is None:
            return torch.matmul(final, self.item_emb.weight.transpose(0, 1))
        cand_emb = self.item_emb(candidate_items)
        return (cand_emb * final.unsqueeze(1)).sum(dim=-1)


class BehaviorAwareSASRec(SASRec):
    def __init__(self, n_items, n_behaviors=3, hidden_size=64, max_len=50, n_blocks=2, n_heads=1, dropout=0.2):
        super().__init__(n_items, hidden_size, max_len, n_blocks, n_heads, dropout)
        self.behavior_emb = nn.Embedding(n_behaviors + 1, hidden_size, padding_idx=0)
        nn.init.normal_(self.behavior_emb.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.behavior_emb.weight[0].fill_(0)

    def encode(self, item_ids, behavior_ids=None):
        if behavior_ids is None:
            raise ValueError("BehaviorAwareSASRec requires behavior_ids")
        bsz, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_emb(item_ids) + self.behavior_emb(behavior_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)


def pad_history(events: Sequence[BehaviorEvent], max_len: int):
    items = [e.item_id for e in events][-max_len:]
    behs = [e.behavior_id for e in events][-max_len:]
    pad = max_len - len(items)
    if pad > 0:
        items = [0] * pad + items
        behs = [BEHAVIOR_PAD] * pad + behs
    return items, behs


class NextPurchaseDataset(Dataset):
    """Next-purchase instances from a multi-behavior (or purchase-only) history."""

    def __init__(self, histories, n_items, max_len=50, n_negatives=1, seed=42, max_users=None):
        self.n_items = n_items
        self.max_len = max_len
        self.n_negatives = n_negatives
        self.rng = np.random.default_rng(seed)
        self.examples = []
        users = list(histories.keys())
        if max_users is not None:
            users = users[:max_users]
        for user in users:
            seq = histories[user]
            buy_positions = [i for i, e in enumerate(seq) if e.behavior_id == BEHAVIOR_BUY]
            for bpos in buy_positions:
                if bpos == 0:
                    continue
                self.examples.append({
                    "user_id": int(user),
                    "history": seq[:bpos],
                    "target_item": seq[bpos].item_id,
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        items, behs = pad_history(ex["history"], self.max_len)
        banned = {e.item_id for e in ex["history"] if e.behavior_id == BEHAVIOR_BUY}
        banned.add(ex["target_item"])
        negs = []
        while len(negs) < self.n_negatives:
            neg = int(self.rng.integers(1, self.n_items + 1))
            if neg not in banned:
                negs.append(neg)
        return {
            "item_ids": torch.tensor(items, dtype=torch.long),
            "behavior_ids": torch.tensor(behs, dtype=torch.long),
            "target_item": torch.tensor(ex["target_item"], dtype=torch.long),
            "neg_items": torch.tensor(negs, dtype=torch.long),
        }


def collate(batch):
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0]}


def bce_ranking_loss(model, item_ids, behavior_ids, pos_items, neg_items):
    # LEFT-padded inputs: use the last timestep representation.
    seq_emb = model.encode(item_ids, behavior_ids)
    final = seq_emb[:, -1, :]
    pos_logits = (final * model.item_emb(pos_items)).sum(dim=-1)
    neg_emb = model.item_emb(neg_items)
    if neg_emb.dim() == 2:
        neg_logits = (final * neg_emb).sum(dim=-1)
    else:
        neg_logits = (neg_emb * final.unsqueeze(1)).sum(dim=-1)
    return -F.logsigmoid(pos_logits).mean() - F.logsigmoid(-neg_logits).mean()


@dataclass
class TrainConfig:
    hidden_size: int = 64
    max_len: int = 50
    n_blocks: int = 2
    n_heads: int = 1
    dropout: float = 0.2
    batch_size: int = 128
    lr: float = 1e-3
    epochs: int = 10
    n_negatives: int = 1
    seed: int = 42
    device: str = "cpu"
    patience: int = 3


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_model(mode: str, n_items: int, cfg: TrainConfig):
    kwargs = dict(
        n_items=n_items,
        hidden_size=cfg.hidden_size,
        max_len=cfg.max_len,
        n_blocks=cfg.n_blocks,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
    )
    if mode == "purchase_only":
        return SASRec(**kwargs)
    if mode == "behavior_aware":
        return BehaviorAwareSASRec(n_behaviors=3, **kwargs)
    raise ValueError(mode)


print("Model helpers ready")


## 5. Shared-candidate scoring

Candidates are **original product IDs**. We map with shared `item_to_idx`, score in that list order, and pass the raw ID lists + aligned scores to `evaluate_model_dataset`.


In [ ]:
@torch.no_grad()
def score_shared_candidates(model, histories, candidate_sets, item_to_idx, max_len, device, batch_size=256):
    """Return {user: [score,...]} aligned with candidate_sets[user] order (original IDs)."""
    model.eval()
    users = [u for u in candidate_sets.keys() if u in histories and histories[u]]
    scores_out = {}

    for start in tqdm(range(0, len(users), batch_size), desc="score candidates", leave=False):
        batch_users = users[start:start + batch_size]
        item_rows, beh_rows, cand_rows, keep_users = [], [], [], []
        for u in batch_users:
            cands = candidate_sets[u]
            try:
                cand_idx = [item_to_idx[c] for c in cands]
            except KeyError:
                continue
            items, behs = pad_history(histories[u], max_len)
            item_rows.append(items)
            beh_rows.append(behs)
            cand_rows.append(cand_idx)
            keep_users.append(u)
        if not keep_users:
            continue
        item_ids = torch.tensor(item_rows, dtype=torch.long, device=device)
        behavior_ids = torch.tensor(beh_rows, dtype=torch.long, device=device)
        cand_items = torch.tensor(cand_rows, dtype=torch.long, device=device)
        batch_scores = model.predict(item_ids, behavior_ids, candidate_items=cand_items)
        for i, u in enumerate(keep_users):
            scores_out[u] = batch_scores[i].detach().cpu().tolist()
    return scores_out


def evaluate_on_shared_split(model, histories, split="val"):
    if split == "val":
        cands, targets, users = val_candidate_sets, val_targets, val_eval_users
    elif split == "test":
        cands, targets, users = test_candidate_sets, test_targets, test_eval_users
    else:
        raise ValueError(split)

    # Restrict to users we can score (have history + candidates)
    cands = {u: cands[u] for u in users if u in cands and u in histories}
    model_scores = score_shared_candidates(
        model, histories, cands, item_to_idx, MAX_SEQ_LEN, DEVICE
    )
    # evaluate_model_dataset requires identical user keys
    cands = {u: cands[u] for u in model_scores}
    targets_sub = {u: targets[u] for u in model_scores}
    summary, user_results = evaluate_model_dataset(
        candidate_sets=cands,
        model_scores=model_scores,
        targets=targets_sub,
        k=EVAL_K,
    )
    return summary, user_results


def warm_cold_breakdown(user_results, train_sequences):
    train_items = {item for seq in train_sequences.values() for item in seq}
    df = user_results.copy()
    df["target_seen_in_train"] = df["target_item"].isin(train_items)
    rows = (
        df.groupby("target_seen_in_train")
        .agg(
            n_users=("user_id", "count"),
            HitRate_at_10=("HitRate@10", "mean"),
            NDCG_at_10=("NDCG@10", "mean"),
            MRR=("RR", "mean"),
        )
        .reset_index()
    )
    rows["target_group"] = rows["target_seen_in_train"].map(
        {True: "Seen in training (warm)", False: "Unseen in training (cold)"}
    )
    return rows[["target_group", "n_users", "HitRate_at_10", "NDCG_at_10", "MRR"]]


print("Shared scoring helpers ready")


## 6. Train


In [ ]:
cfg = TrainConfig(
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_SEQ_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    n_negatives=N_NEGATIVES_TRAIN,
    seed=SEED,
    device=DEVICE,
    patience=PATIENCE,
)
print(cfg)


def train_one(mode: str, histories: Dict[int, List[BehaviorEvent]]):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)
    model = build_model(mode, n_items, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    ds = NextPurchaseDataset(
        histories,
        n_items=n_items,
        max_len=cfg.max_len,
        n_negatives=cfg.n_negatives,
        seed=cfg.seed,
        max_users=MAX_TRAIN_USERS,
    )
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate)
    print(f"[{mode}] train examples: {len(ds):,}")

    best_metric = -1.0
    best_path = OUTPUT_DIR / mode / f"{mode}_best.pt"
    best_path.parent.mkdir(parents=True, exist_ok=True)
    patience_left = cfg.patience
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        losses = []
        for batch in tqdm(loader, desc=f"{mode} epoch {epoch}", leave=False):
            item_ids = batch["item_ids"].to(device)
            behavior_ids = batch["behavior_ids"].to(device)
            pos = batch["target_item"].to(device)
            neg = batch["neg_items"].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = bce_ranking_loss(model, item_ids, behavior_ids, pos, neg)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        # Early-stop on shared validation candidates
        summary, _ = evaluate_on_shared_split(model, histories, split="val")
        mean_loss = float(np.mean(losses)) if losses else None
        history.append({"epoch": epoch, "loss": mean_loss, "val": summary})
        print(
            f"[{mode}] epoch={epoch} loss={mean_loss:.4f} "
            f"val HitRate@10={summary['HitRate@10']:.4f} "
            f"NDCG@10={summary['NDCG@10']:.4f} MRR={summary['MRR']:.4f} "
            f"n={summary['n_users']}"
        )
        metric = summary["NDCG@10"]
        if metric > best_metric:
            best_metric = metric
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "mode": mode,
                    "n_items": n_items,
                    "config": asdict(cfg),
                    "val_summary": summary,
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{mode}] early stopping at epoch {epoch}")
                break

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    return model, best_path, history


modes = []
if MODE == "both":
    modes = ["purchase_only", "behavior_aware"]
else:
    modes = [MODE]

trained = {}
for mode in modes:
    if mode == "purchase_only":
        hist = purchase_histories
    else:
        if behavior_histories is None:
            print("Skipping behavior_aware (no cart histories available)")
            continue
        hist = behavior_histories
    print(f"\n=== Training {mode} ===")
    model, path, hist_log = train_one(mode, hist)
    trained[mode] = {"model": model, "histories": hist, "checkpoint": path, "history": hist_log}

print("Trained:", list(trained.keys()))


## 7. Evaluate on shared candidates


In [ ]:
all_summaries = []
all_breakdowns = {}

for mode, payload in trained.items():
    print(f"\n=== {mode} @ {EVAL_SPLIT} ===")
    summary, user_results = evaluate_on_shared_split(
        payload["model"], payload["histories"], split=EVAL_SPLIT
    )
    summary_row = {"model": mode, **summary}
    all_summaries.append(summary_row)
    print(pd.DataFrame([summary_row]))

    breakdown = warm_cold_breakdown(user_results, train_sequences)
    all_breakdowns[mode] = breakdown
    print("Warm / cold breakdown:")
    display(breakdown)

    out_csv = OUTPUT_DIR / mode / f"{mode}_{EVAL_SPLIT}_user_results.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    user_results.to_csv(out_csv, index=False)
    print("Wrote", out_csv)

comparison = pd.DataFrame(all_summaries).set_index("model")
print("\nComparison:")
display(comparison)

summary_path = OUTPUT_DIR / f"summary_{EVAL_SPLIT}.json"
with summary_path.open("w") as f:
    json.dump(
        {
            "summaries": all_summaries,
            "breakdowns": {m: df.to_dict(orient="records") for m, df in all_breakdowns.items()},
            "eval_split": EVAL_SPLIT,
            "max_seq_len": MAX_SEQ_LEN,
            "val_cutoff": str(VAL_CUTOFF),
            "test_cutoff": str(TEST_CUTOFF),
            "n_items": n_items,
            "n_negatives_eval": N_NEGATIVES_EVAL,
        },
        f,
        indent=2,
    )
print("Wrote", summary_path.resolve())


## Notes

- Do **not** regenerate candidate sets here — always load `artifacts/shared_evaluation_candidates.pkl`.
- Scores must follow each candidate list’s existing order; `evaluation_utils` handles ranking.
- Primary comparison should use the same `val_eval_users` / `test_eval_users` as the popularity baseline.
- Prefer `EVAL_SPLIT="val"` while developing; reserve test for the final report.
- `behavior_aware` needs local `*_subsampled.parquet` files; `purchase_only` can train from the shared pickle alone.

- Sequences are **left-padded**; the model always reads the representation at the last timestep (`seq_emb[:, -1]`), matching standard SASRec.
